In [7]:
import pandas as pd

# 1) Load your data (detect .csv vs .xlsx)
INPUT_FILE = "guardian_microsoft_articles.csv"  
#INPUT_FILE  = "apple_articles_with_body.xlsx"
if INPUT_FILE.lower().endswith((".xls", ".xlsx")):
    df = pd.read_excel(INPUT_FILE)
else:
    df = pd.read_csv(INPUT_FILE)

# 2) Inspect your columns
print("Available columns:", df.columns.tolist())


Available columns: ['Body', 'Headline', 'Subheadline', 'Date', 'URL', 'Authors']


In [10]:
import pandas as pd
from langchain_community.llms import LlamaCpp

# 1) Load your data (handles both .csv and .xlsx)
INPUT_FILE  = "guardian_microsoft_articles.csv"
if INPUT_FILE.lower().endswith((".xls", ".xlsx")):
    df = pd.read_excel(INPUT_FILE)
else:
    df = pd.read_csv(INPUT_FILE)

# 2) Set the column name that contains your article text
TEXT_COL    = "Body"
OUTPUT_CSV  = "sentiment_output_microsoft.csv"

# 3) Initialize your local Llama 3 model
model_path = "Llama-2-7B-Chat-GGUF/Llama-2-7B-Chat-GGUF/llama-2-7b-chat.Q4_K_M.gguf"
llm = LlamaCpp(
    model_path=model_path,
    temperature=0.0,    # deterministic output
    max_tokens=64,
    top_p=0.95,
    n_ctx=4096,
    verbose=False,
)

# 4) Helper to get sentiment (with truncation)
MAX_CHARS = 4096   # adjust as needed (2 048 chars ~ 2 048 tokens approx)
def get_sentiment(text: str) -> str:
    snippet = text[:MAX_CHARS]  # never send more than MAX_CHARS
    prompt = (
        "Classify the sentiment of the following text as positive or negative.\n\n"
        f"\"\"\"\n{snippet}\n\"\"\"\n\nSentiment:"
    )
    resp = llm(prompt)
    label = resp.strip().split()[0].lower()
    return label if label in ("positive", "negative") else "neutral"

# 5) Apply to each article
df["sentiment"] = (
    df[TEXT_COL]
    .fillna("")           # replace NaN with empty string
    .astype(str)
    .apply(get_sentiment)
)

# 6) Write out the augmented CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Wrote {len(df)} rows with sentiment to {OUTPUT_CSV}")


llama_init_from_model: n_batch is less than GGML_KQ_MASK_PAD - increasing to 64
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mm_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h64           (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h80           (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h96           (not supported)
ggml_me

✅ Wrote 22 rows with sentiment to sentiment_output_microsoft.csv


In [6]:
response = llm.invoke("Explain how the transformer architecture works in machine learning, in simple terms.")
print(response)

Llama.generate: 4 prefix-match hit, remaining 13 prompt tokens to eval
llama_perf_context_print:        load time =    2720.09 ms
llama_perf_context_print: prompt eval time =    1022.67 ms /    13 tokens (   78.67 ms per token,    12.71 tokens per second)
llama_perf_context_print:        eval time =   36581.94 ms /   511 runs   (   71.59 ms per token,    13.97 tokens per second)
llama_perf_context_print:       total time =   37833.14 ms /   524 tokens




Transformers are a type of neural network architecture introduced in 2017 by Vaswani et al. in the paper "Attention is All You Need". They have since become one of the most popular architectures in natural language processing (NLP) tasks, such as language translation and text generation.
In this answer, we'll explain how transformers work in simple terms, including their architecture, key components, and training process.
**What is a Transformer?**
A transformer is a type of neural network that processes input sequences of variable length. Unlike traditional recurrent neural networks (RNNs), which process sequences one element at a time, transformers process the entire sequence in parallel. This makes them much faster and more scalable than RNNs for long sequences.
**Transformer Architecture**
The transformer architecture consists of several components:
1. Encoder: The encoder takes in a sequence of tokens (e.g., words or characters) and outputs a continuous representation of the inp